In [ ]:
import numpy as np
import pandas as pd

np.random.seed(42)
N = 60000  # Number of admissions

# ─── Demographic Features ────────────────────────────────────────────────────
age = np.random.normal(62, 17, N).clip(18, 95)
gender = np.random.choice([0, 1], N, p=[0.46, 0.54])  # 0=M, 1=F
ethnicity = np.random.choice(
    ['WHITE', 'BLACK', 'HISPANIC', 'ASIAN', 'OTHER'],
    N, p=[0.68, 0.12, 0.09, 0.06, 0.05]
)
insurance = np.random.choice(
    ['Medicare', 'Medicaid', 'Private', 'Self Pay', 'Government'],
    N, p=[0.42, 0.17, 0.33, 0.05, 0.03]
)
marital_status = np.random.choice(
    ['MARRIED', 'SINGLE', 'WIDOWED', 'DIVORCED', 'UNKNOWN'],
    N, p=[0.48, 0.25, 0.13, 0.10, 0.04]
)
admission_type = np.random.choice(
    ['EMERGENCY', 'ELECTIVE', 'URGENT', 'NEWBORN'],
    N, p=[0.52, 0.29, 0.16, 0.03]
)

# ─── Clinical Features ────────────────────────────────────────────────────────
los_days = np.random.exponential(5.5, N).clip(0.5, 90)
icu_flag = np.random.binomial(1, 0.38, N)
num_diagnoses = np.random.poisson(8, N).clip(1, 30)
num_procedures = np.random.poisson(4, N).clip(0, 25)
charlson_score = np.random.poisson(2.5, N).clip(0, 12)
elixhauser_score = np.random.normal(3.2, 2.1, N).clip(0, 15).astype(int)
num_medications = np.random.poisson(10, N).clip(1, 40)
prior_admissions = np.random.poisson(1.8, N).clip(0, 15)
prior_mh_admissions = np.random.poisson(0.6, N).clip(0, 8)

# ─── Lab Values (normalized) ─────────────────────────────────────────────────
sodium = np.random.normal(138, 4.5, N).clip(120, 155)
potassium = np.random.normal(4.0, 0.6, N).clip(2.5, 6.5)
bun = np.random.exponential(20, N).clip(5, 120)
creatinine = np.random.exponential(1.1, N).clip(0.4, 10)
glucose = np.random.normal(120, 40, N).clip(50, 500)
wbc = np.random.normal(10, 4, N).clip(1, 40)
hemoglobin = np.random.normal(11.5, 2.5, N).clip(4, 18)
albumin = np.random.normal(3.2, 0.7, N).clip(1.5, 5.0)

# ─── Vital Signs ─────────────────────────────────────────────────────────────
heart_rate_mean = np.random.normal(82, 16, N).clip(40, 160)
sbp_mean = np.random.normal(122, 22, N).clip(70, 200)
spo2_mean = np.random.normal(96, 3, N).clip(80, 100)
resp_rate_mean = np.random.normal(18, 5, N).clip(8, 45)
temp_mean = np.random.normal(37.0, 0.8, N).clip(34, 41)

# ─── Mental Health Specific ───────────────────────────────────────────────────
prev_psych_meds = np.random.binomial(1, 0.22, N)
substance_use_hx = np.random.binomial(1, 0.18, N)
trauma_hx = np.random.binomial(1, 0.12, N)
social_support_poor = np.random.binomial(1, 0.28, N)
sleep_disorder = np.random.binomial(1, 0.15, N)
pain_score_mean = np.random.exponential(3.5, N).clip(0, 10)

# ─── NLP features (simulated TF-IDF scores) ──────────────────────────────────
note_depression_score = np.random.beta(1.5, 8, N)  # Low prevalence
note_anxiety_score = np.random.beta(1.5, 7, N)
note_psychosis_score = np.random.beta(1, 15, N)
note_suicide_risk = np.random.beta(0.8, 20, N)

# ─── Build DataFrame ─────────────────────────────────────────────────────────
df = pd.DataFrame({
    'hadm_id': range(100000, 100000 + N),
    'subject_id': np.random.randint(1, 46522, N),
    'age': age,
    'gender': gender,
    'ethnicity': ethnicity,
    'insurance': insurance,
    'marital_status': marital_status,
    'admission_type': admission_type,
    'los_days': los_days,
    'icu_flag': icu_flag,
    'num_diagnoses': num_diagnoses,
    'num_procedures': num_procedures,
    'charlson_score': charlson_score,
    'elixhauser_score': elixhauser_score,
    'num_medications': num_medications,
    'prior_admissions': prior_admissions,
    'prior_mh_admissions': prior_mh_admissions,
    'sodium': sodium,
    'potassium': potassium,
    'bun': bun,
    'creatinine': creatinine,
    'glucose': glucose,
    'wbc': wbc,
    'hemoglobin': hemoglobin,
    'albumin': albumin,
    'heart_rate_mean': heart_rate_mean,
    'sbp_mean': sbp_mean,
    'spo2_mean': spo2_mean,
    'resp_rate_mean': resp_rate_mean,
    'temp_mean': temp_mean,
    'prev_psych_meds': prev_psych_meds,
    'substance_use_hx': substance_use_hx,
    'trauma_hx': trauma_hx,
    'social_support_poor': social_support_poor,
    'sleep_disorder': sleep_disorder,
    'pain_score_mean': pain_score_mean,
    'note_depression_score': note_depression_score,
    'note_anxiety_score': note_anxiety_score,
    'note_psychosis_score': note_psychosis_score,
    'note_suicide_risk': note_suicide_risk,
})

# ─── Generate Target Variables with realistic correlations ────────────────────
# Mental health diagnosis (primary outcome)
mh_logit = (
    -2.5
    + 0.6 * df['prev_psych_meds']
    + 0.5 * df['substance_use_hx']
    + 0.4 * df['prior_mh_admissions'] / 3
    + 0.3 * df['trauma_hx']
    + 0.25 * df['social_support_poor']
    + 0.2 * (df['age'] - 62) / 17
    + 0.3 * df['note_depression_score'] * 5
    + 0.25 * df['note_anxiety_score'] * 5
    + 0.4 * df['note_suicide_risk'] * 8
    + 0.15 * df['pain_score_mean'] / 5
    + 0.1 * (df['gender'] == 1).astype(float)
    + np.random.normal(0, 0.5, N)
)
df['has_mental_health_dx'] = (1 / (1 + np.exp(-mh_logit)) > 0.33).astype(int)


# Mortality
df['hospital_expire_flag'] = np.random.binomial(1, 0.085, N)
print(f' Dataset created: {df.shape[0]:,} admissions \u00d7 {df.shape[1]} features')
print(f'\nTarget distributions:')
print(f'  Mental Health Diagnosis: {df["has_mental_health_dx"].mean():.1%}')
print(f'  In-Hospital Mortality:   {df["hospital_expire_flag"].mean():.1%}')
df.head(10)

 Dataset created: 60,000 admissions × 42 features

Target distributions:
  Mental Health Diagnosis: 18.0%
  In-Hospital Mortality:   8.6%


,hadm_id,subject_id,age,gender,ethnicity,insurance,marital_status,admission_type,los_days,icu_flag,...,trauma_hx,social_support_poor,sleep_disorder,pain_score_mean,note_depression_score,note_anxiety_score,note_psychosis_score,note_suicide_risk,has_mental_health_dx,hospital_expire_flag
0,100000,27227,70.444141,0,OTHER,Private,MARRIED,EMERGENCY,0.860933,1,...,0,0,0,10.000000,0.173731,0.313295,0.007444,0.126328,0,0
1,100001,22096,59.649507,1,WHITE,Private,SINGLE,EMERGENCY,9.246672,1,...,0,0,0,10.000000,0.107533,0.044131,0.074333,0.058402,1,0
2,100002,38859,73.010705,1,WHITE,Medicare,MARRIED,URGENT,0.818404,1,...,0,1,0,1.052304,0.057512,0.076421,0.022595,0.098133,1,0
3,100003,21055,87.891508,1,BLACK,Private,MARRIED,EMERGENCY,12.281800,0,...,0,0,0,8.483931,0.048523,0.114711,0.055725,0.008262,0,0
4,100004,11579,58.019393,0,WHITE,Private,MARRIED,EMERGENCY,10.902987,0,...,0,0,0,3.457572,0.068658,0.167851,0.074256,0.129100,0,0
5,100005,15628,58.019672,1,BLACK,Medicaid,MARRIED,EMERGENCY,3.907345,1,...,0,0,0,6.305632,0.099289,0.059909,0.034615,0.016452,0,0
6,100006,3564,88.846618,1,WHITE,Medicare,DIVORCED,EMERGENCY,3.271828,1,...,0,1,1,0.767635,0.129762,0.401613,0.001584,0.002559,1,0
7,100007,18436,75.046390,0,BLACK,Medicaid,WIDOWED,ELECTIVE,4.800822,1,...,0,0,0,5.203169,0.335130,0.111280,0.078998,0.000809,1,0
8,100008,43640,54.018935,0,WHITE,Self Pay,MARRIED,EMERGENCY,13.621472,1,...,0,0,1,0.889367,0.650783,0.387982,0.234492,0.088979,1,0
9,100009,42480,71.223521,1,WHITE,Self Pay,SINGLE,EMERGENCY,13.339405,1,...,0,0,0,10.000000,0.097497,0.234718,0.000385,0.036103,1,0


In [ ]:
# ─── Dataset Overview ─────────────────────────────────────────────────────────
print('=== DATASET SUMMARY ===')
print(f'Total admissions:              {len(df):,}')
print(f'Mental health diagnoses:       {df["has_mental_health_dx"].sum():,} ({df["has_mental_health_dx"].mean():.1%})')
print(f'In-hospital deaths:            {df["hospital_expire_flag"].sum():,} ({df["hospital_expire_flag"].mean():.1%})')
print(f'ICU patients:                  {df["icu_flag"].sum():,} ({df["icu_flag"].mean():.1%})')
print(f'\nAge: Mean={df["age"].mean():.1f}, Median={df["age"].median():.1f}, Std={df["age"].std():.1f}')
print(f'LOS: Mean={df["los_days"].mean():.1f}d, Median={df["los_days"].median():.1f}d')
print(f'\nMissing values: {df.isnull().sum().sum()}')

=== DATASET SUMMARY ===
Total admissions:              60,000
Mental health diagnoses:       10,777 (18.0%)
In-hospital deaths:            5,132 (8.6%)
ICU patients:                  22,943 (38.2%)

Age: Mean=61.9, Median=62.0, Std=16.6
LOS: Mean=5.5d, Median=3.8d

Missing values: 0


In [ ]:
cfrom google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
output_path = '/content/drive/MyDrive/csv/mental_health_patient_data.csv'
df.to_csv(output_path, index=False)
print(f'DataFrame successfully saved to {output_path}')

DataFrame successfully saved to /content/drive/MyDrive/csv/mental_health_patient_data.csv
